# 第2章：追踪 LLaVA 的图文数据流

**目标：用运行证据解释 576 个视觉向量如何进入 LLM。**

沿用第1章的 `llava-week1` 环境：Transformers **4.57.1**、`llava-hf/llava-1.5-7b-hf` 及已记录的模型 commit。运行地点仍是**云端 Linux GPU 的 JupyterLab**；本机只需编辑文件。

建议分三次学习：① 第 0–5 节，处理器与视觉特征；② 第 6–8 节，真实融合与生成；③ 第 9–12 节，遮挡对照、连接模块比较和学习记录。约 8–12 小时，按理解程度调整。

本 notebook 独立运行，不依赖第1章留在内存中的变量。先关闭第1章的 GPU 内核（只关标签页通常不会释放模型），选择 **Python (llava-week1)**，从头逐格执行。首次加载模型前确认 GPU 空闲。无需安装原生 LLaVA、MiniGPT-4、DeepSpeed 或新建训练环境。

这份材料只做推理与观察，没有参数更新。代码不会预填模型回答或实验结论。**本地完成 notebook 结构、Python 语法和小型 CPU 逻辑检查；7B 模型的实际 GPU 运行仍需你在云端验证。**

参考：[第1章报告](../experiments/chapter1-baseline.md)、[对应版本文档](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/llava)、[HF 模型源码](https://github.com/huggingface/transformers/blob/v4.57.1/src/transformers/models/llava/modeling_llava.py)、[processor 源码](https://github.com/huggingface/transformers/blob/v4.57.1/src/transformers/models/llava/processing_llava.py)。本章以**实际安装版本的函数**为准。

学习按章节推进；`llava-week1` 是已经安装的环境与内核名，继续沿用即可。历史输出中的原文件名、目录名和日期保留为运行记录。


## 0. 先设置下载地址与缓存，再导入模型库

这一格要在新内核先执行，避免环境变量在导入后才设置而没有生效。沿用第1章的镜像地址；需要官方地址时修改 `HF_ENDPOINT_URL`。公开模型不需要在代码里填写 token。

`CACHE_DIR=None` 使用默认 / 已配置的 Hugging Face 缓存。如果第1章的权重存在其他位置，填写你实际使用的 **hub 缓存目录**，例如 `/root/autodl-tmp/huggingface/hub`。不要仅为本章换一个空缓存目录。`LOCAL_FILES_ONLY=True` 可限制为只读已有缓存；首次缓存未齐时设为 False。


In [ ]:
import os
import sys

HF_ENDPOINT_URL = "https://hf-mirror.com"
CACHE_DIR = None
LOCAL_FILES_ONLY = False

if "transformers" in sys.modules or "huggingface_hub" in sys.modules:
    raise RuntimeError("请先 Restart Kernel，再从这一格开始；下载配置应先于模型库导入。")
os.environ["OMP_NUM_THREADS"] = "1"
os.environ["HF_ENDPOINT"] = HF_ENDPOINT_URL
os.environ["HF_HUB_DISABLE_IMPLICIT_TOKEN"] = "1"
os.environ["HF_HUB_ETAG_TIMEOUT"] = "30"
os.environ["HF_HUB_DOWNLOAD_TIMEOUT"] = "120"
print("Python:", sys.executable)
print("HF endpoint:", HF_ENDPOINT_URL)
print("cache_dir:", CACHE_DIR, "local_files_only:", LOCAL_FILES_ONLY)


## 1. 配置图片、问题与独立输出目录

使用第1章的两位工作人员图片，问题改为不预设两人都坐着的版本。后面的视觉特征、融合与生成都使用同一张图和同一问题。

每次执行本格新建 `outputs/ch2-时间戳/`，不会覆盖第1章。若只是重跑某个后续实验，不必重跑配置格。模型版本固定为第1章的 commit，**不再使用 main**。如果缺少依赖，请先在终端激活已有 `llava-week1` 环境，按 `src/requirements.txt` 检查，安装后重启内核。


In [ ]:
import json
import time
import hashlib
import inspect
import subprocess
from contextlib import contextmanager
from datetime import datetime, timezone
from pathlib import Path

import torch
import transformers
from PIL import Image, ImageOps, ImageDraw
from IPython.display import display, Markdown
from transformers import AutoProcessor, LlavaForConditionalGeneration

assert transformers.__version__ == "4.57.1", "请选择 llava-week1 内核，勿在本章随意升级 Transformers。"
assert torch.cuda.is_available(), "本 notebook 的实际模型部分需要云端 CUDA GPU。"

# 如自动定位失败，显式填写你的云端仓库绝对路径。
PROJECT_ROOT = next(
    (p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
     if (p / "src" / "chapter1_llava_baseline.ipynb").is_file()),
    None,
)
assert PROJECT_ROOT is not None, "请先把 PROJECT_ROOT 改为云端仓库路径。"
IMAGE_PATH = PROJECT_ROOT / "data/my_vlm_test_set/images/000000008211.jpg"
MODEL_ID = "llava-hf/llava-1.5-7b-hf"
MODEL_REVISION = "b234b804b114d9e37bb655e11cbbb5f5e971b7a9"
QUESTION = "图中两位工作人员的上衣主要是什么颜色？"
QA_INSTRUCTION = "请仅根据图片中的可见信息回答问题。回答简洁明确；如果图片无法支持确定答案，请说明无法确定。不要猜测图片外的信息。"
DEVICE = "cuda:0"
DTYPE = torch.float16
SEED = 42
GENERATION = {"max_new_tokens": 64, "do_sample": False, "num_beams": 1, "use_cache": True}
torch.manual_seed(SEED)
torch.cuda.set_device(0)
assert IMAGE_PATH.is_file(), f"找不到图片：{IMAGE_PATH}"

def stamp():
    return datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S.%fZ")

def save_json(path, value):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(value, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")

def sha256_file(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()

def git_output(*args, cwd=PROJECT_ROOT):
    result = subprocess.run(["git", *args], cwd=cwd, capture_output=True, text=True, timeout=30)
    return result.stdout.strip() if result.returncode == 0 else None

RUN_DIR = PROJECT_ROOT / "outputs" / ("ch2-" + stamp())
RUN_DIR.mkdir(parents=True, exist_ok=False)
save_json(RUN_DIR / "config.json", {
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "image": str(IMAGE_PATH.relative_to(PROJECT_ROOT)), "image_sha256": sha256_file(IMAGE_PATH),
    "question": QUESTION, "instruction": QA_INSTRUCTION, "generation": GENERATION,
    "seed": SEED, "dtype": str(DTYPE), "device": DEVICE, "batch_size": 1,
    "hf_endpoint": HF_ENDPOINT_URL, "cache_dir": CACHE_DIR,
})
print("根目录:", PROJECT_ROOT)
print("输出目录:", RUN_DIR)
print(subprocess.run(["nvidia-smi"], capture_output=True, text=True, check=True).stdout)


## 2. 只加载一次模型，保存实际环境

processor 的 patch size、CLS 数量和视觉特征选择策略必须与模型一致。本 notebook 针对 **LLaVA 1.5 的 CLIP 单图路径**；换成其他模型、any-resolution 或多图版本时，需要重新检查这些假设。

OOM 时先关闭旧 GPU 内核或重启当前内核；不要连续执行加载格来尝试解决。模型加载后再执行同一格会明确报错，避免同时留下两份模型。


In [ ]:
if "model" in globals():
    raise RuntimeError("model 已存在：继续后面的格子即可；需要重新加载时先 Restart Kernel。")

load_args = {"revision": MODEL_REVISION, "local_files_only": LOCAL_FILES_ONLY}
if CACHE_DIR is not None:
    load_args["cache_dir"] = str(CACHE_DIR)
started = time.perf_counter()
processor = AutoProcessor.from_pretrained(MODEL_ID, **load_args)
model = LlavaForConditionalGeneration.from_pretrained(
    MODEL_ID, **load_args, dtype=DTYPE, device_map={"": DEVICE}, low_cpu_mem_usage=True,
)
model.eval()
processor.tokenizer.padding_side = "left"
processor.patch_size = model.config.vision_config.patch_size
processor.vision_feature_select_strategy = model.config.vision_feature_select_strategy
processor.num_additional_image_tokens = 1  # 本 checkpoint 的 CLIP 包含一个 CLS token
assert model.config.vision_config.model_type == "clip_vision_model"
assert model.config.vision_feature_select_strategy == "default"
assert isinstance(model.config.vision_feature_layer, int)
torch.cuda.synchronize()

gpu = torch.cuda.get_device_properties(0)
environment = {
    "recorded_at_utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version, "executable": sys.executable,
    "torch": torch.__version__, "transformers": transformers.__version__,
    "cuda_runtime": torch.version.cuda, "gpu": gpu.name,
    "gpu_total_gib": gpu.total_memory / 1024**3,
    "model_id": MODEL_ID, "model_revision": MODEL_REVISION,
    "repository_commit": git_output("rev-parse", "HEAD"),
    "repository_status": git_output("status", "--short"),
    "load_seconds": time.perf_counter() - started,
    "allocated_gib_after_load": torch.cuda.memory_allocated() / 1024**3,
}
save_json(RUN_DIR / "environment.json", environment)
freeze = subprocess.run([sys.executable, "-m", "pip", "freeze"], capture_output=True, text=True, check=True)
(RUN_DIR / "requirements-freeze.txt").write_text(freeze.stdout, encoding="utf-8")
save_json(RUN_DIR / "model-config.json", model.config.to_dict())
save_json(RUN_DIR / "generation-config.json", model.generation_config.to_dict())
print(json.dumps(environment, ensure_ascii=False, indent=2))
print("视觉层:", model.config.vision_feature_layer,
      "patch size:", processor.patch_size,
      "语言维度:", model.config.text_config.hidden_size)


## 3. 从原图、聊天模板到张量

`image.size` 是 `(宽,高)`，`pixel_values` 的后两维是 `(高,宽)`。原图尺寸不等于进入视觉编码器的分辨率。先看实际 processor 配置和原图，不要用原图面积推算视觉 token 数。

聊天模板字符串里通常只有一个 `<image>`，processor 会根据预处理后的尺寸把它展开。以下代码不把参考答案或评分规则送给模型。


In [ ]:
def open_rgb(path):
    with Image.open(path) as source:
        return ImageOps.exif_transpose(source).convert("RGB")

def prepare_inputs(pil_image, question):
    full_question = question.strip() + "\n" + QA_INSTRUCTION
    conversation = [{"role": "user", "content": [
        {"type": "image"}, {"type": "text", "text": full_question},
    ]}]
    prompt = processor.apply_chat_template(conversation, add_generation_prompt=True)
    batch = processor(images=pil_image, text=prompt, return_tensors="pt", truncation=False)
    return prompt, dict(batch)

def to_device(batch):
    return {name: value.to(device=DEVICE, dtype=DTYPE if value.is_floating_point() else value.dtype)
            for name, value in batch.items()}

image = open_rgb(IMAGE_PATH)
display(image)
print("原图宽高:", image.size)
print("resize / crop 配置:", processor.image_processor.size, processor.image_processor.crop_size)
prompt, inputs_cpu = prepare_inputs(image, QUESTION)
print("模板文本:\n", prompt)
for name, tensor in inputs_cpu.items():
    print(name, tuple(tensor.shape), tensor.dtype)
save_json(RUN_DIR / "processor-config.json", processor.image_processor.to_dict())


## 4. 验证图像 token 数；观察问题长度变化

本 checkpoint 去掉 CLS 后，每个 patch 对应一个视觉向量。按**实际** `pixel_values` 与 `patch_size` 计算数量，并与 `input_ids` 中的图像 token 数核对。

用两个不同长度的问题只做预处理，不运行 LLM：图像 token 数应保持不变，完整文本序列长度可能变化。文本直接 tokenizer 与 processor 的长度差，也可以检查占位符是否展开。不要打印全部 576 个图像 token。


In [ ]:
image_token_id = model.config.image_token_id
assert processor.tokenizer.convert_tokens_to_ids(processor.image_token) == image_token_id
height, width = inputs_cpu["pixel_values"].shape[-2:]
expected_image_tokens = ((height // processor.patch_size) * (width // processor.patch_size)
                         + processor.num_additional_image_tokens - 1)
token_mask_cpu = inputs_cpu["input_ids"].eq(image_token_id)
positions = torch.where(token_mask_cpu[0])[0]
assert positions.numel() == expected_image_tokens

token_rows = []
for question in [QUESTION, "请仔细观察画面中的两位工作人员，只回答他们上衣的主要颜色，不描述其他物体。"]:
    candidate_prompt, batch = prepare_inputs(image, question)
    plain_ids = processor.tokenizer(candidate_prompt, return_tensors="pt")["input_ids"]
    count = int(batch["input_ids"].eq(image_token_id).sum())
    plain_count = int(plain_ids.eq(image_token_id).sum())
    assert plain_count == 1 and count == expected_image_tokens
    assert batch["input_ids"].shape[1] - plain_ids.shape[1] == expected_image_tokens - 1
    token_rows.append({"question": question, "image_tokens": count,
                       "sequence_length": batch["input_ids"].shape[1],
                       "tokenizer_only_length": plain_ids.shape[1]})
token_trace = {"pixels": list(inputs_cpu["pixel_values"].shape),
               "expected_image_tokens": expected_image_tokens,
               "image_token_id": image_token_id,
               "image_token_first_position": positions[0].item(),
               "image_token_last_position": positions[-1].item(), "questions": token_rows}
save_json(RUN_DIR / "token-trace.json", token_trace)
print(json.dumps(token_trace, ensure_ascii=False, indent=2))
print("思考：一个占位符展开后，为什么序列长度增加的是 N-1 而不是 N？")


## 5. 读取当前函数，拆开视觉编码与 projector

先定位自己真正执行的代码，记录路径、行号和文件 SHA256；完整函数只在运行时从已安装库读取，保存到本次输出目录。`get_image_features` 在 4.57.1 的此路径返回每张图特征构成的列表；不要假定它始终直接返回三维 tensor。

下一格分别记录包含 CLS 的特征、去掉 CLS 后的特征和投影后的特征。**先读形状，再解释每个轴。** 示例预期约为 `577 → 576` 个向量、`1024 → 4096` 维；以实际配置和断言为准。


In [ ]:
def source_record(label, function):
    function = inspect.unwrap(function)
    lines, start_line = inspect.getsourcelines(function)
    filename = Path(inspect.getsourcefile(function))
    target = RUN_DIR / "installed-source" / (label.replace(".", "_") + ".py")
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text("".join(lines), encoding="utf-8")
    record = {"label": label, "file": str(filename), "start_line": start_line,
              "module_sha256": sha256_file(filename), "saved_source": str(target.relative_to(RUN_DIR))}
    print(f"\n{label} — {filename}:{start_line}")
    print("".join(lines[:100]))
    return record

installed_sources = [source_record(label, function) for label, function in [
    ("LlavaProcessor.__call__", type(processor).__call__),
    ("Projector.forward", type(model.model.multi_modal_projector).forward),
    ("LlavaModel.get_image_features", type(model.model).get_image_features),
    ("LlavaModel.get_placeholder_mask", type(model.model).get_placeholder_mask),
    ("LlavaModel.forward", type(model.model).forward),
]]
save_json(RUN_DIR / "installed-source-index.json", installed_sources)


In [ ]:
inputs_gpu = to_device(inputs_cpu)
with torch.inference_mode():
    vision_out = model.model.vision_tower(inputs_gpu["pixel_values"], output_hidden_states=True)
    with_cls = vision_out.hidden_states[model.config.vision_feature_layer]
    selected = with_cls[:, 1:, :]
    projected = model.model.multi_modal_projector(selected)
    projected_reference = projected.detach().cpu()
    # 一份普通 token embedding 的 CPU 副本，留给第 6 节逐元素核对。
    word_embeddings_cpu = model.get_input_embeddings()(inputs_gpu["input_ids"]).detach().cpu()
    vision_trace = {"pixels": list(inputs_gpu["pixel_values"].shape),
                    "with_cls": list(with_cls.shape), "selected": list(selected.shape),
                    "projected": list(projected.shape),
                    "vision_feature_layer": model.config.vision_feature_layer,
                    "language_hidden_size": model.config.text_config.hidden_size}
assert selected.shape[1] == expected_image_tokens
assert projected.shape[-1] == model.config.text_config.hidden_size
del vision_out, with_cls, selected, projected

# 再走模型自己提供的函数，验证手工拆解的视觉路径。
with torch.inference_mode():
    feature_list = model.model.get_image_features(pixel_values=inputs_gpu["pixel_values"])
    api_features_cpu = torch.stack(feature_list).detach().cpu()
torch.testing.assert_close(api_features_cpu, projected_reference, rtol=1e-2, atol=1e-2)
vision_trace["manual_vs_api_max_abs_diff"] = (api_features_cpu.float() - projected_reference.float()).abs().max().item()
save_json(RUN_DIR / "vision-trace.json", vision_trace)
print(json.dumps(vision_trace, ensure_ascii=False, indent=2))
del feature_list, api_features_cpu


## 6. 本章重点：捕获真实 LLM 输入，手工重建并逐元素核对

`get_input_embeddings()` 先把整个 `input_ids` 映射成向量；其中图像占位符的向量随后被 projector 输出替换。下面使用临时 hook 捕获**同一次实际 forward** 的 projector 输出和语言模型入口 `inputs_embeds`。

手工重建使用布尔索引赋值，而非再次调用模型的融合函数。验证两件事：① 图像位置按顺序装入视觉向量；② 其余位置仍与词嵌入一致。`logits_to_keep=1` 只计算末位置 logits，避免保存整个序列的词表分数。这里不是性能计时。

hook 用 `try/finally` 管理，即使运行报错也会移除，不会在后续生成中重复捕获大张量。CPU 副本只用于观察，不会被送回去替代模型真实输入。


In [ ]:
@contextmanager
def capture_fusion(core):
    captured = {"llm_calls": []}
    handles = []

    def projector_hook(module, args, output):
        captured["projector_input_shape"] = list(args[0].shape)
        captured["projected"] = output.detach().cpu().clone()

    def llm_pre_hook(module, args, kwargs):
        embedding = kwargs.get("inputs_embeds")
        if embedding is None:
            raise RuntimeError("没有在 LLM 入口捕获到 inputs_embeds，请检查实现版本。")
        captured["llm_calls"].append(embedding.detach().cpu().clone())

    try:
        handles.append(core.multi_modal_projector.register_forward_hook(projector_hook))
        handles.append(core.language_model.register_forward_pre_hook(llm_pre_hook, with_kwargs=True))
        yield captured
    finally:
        for handle in handles:
            handle.remove()

def reconstruct_fusion(word_embeddings, image_mask, projected_features):
    # 本练习限定单样本、单图，先检查位置数和向量数，避免静默错位。
    if word_embeddings.ndim != 3 or projected_features.ndim != 3:
        raise ValueError("期望 embeddings 和 projected_features 均为三维。")
    if image_mask.shape != word_embeddings.shape[:2] or word_embeddings.shape[0] != 1:
        raise ValueError("本练习只处理 batch_size=1，且 mask 必须匹配序列。")
    if projected_features.shape[0] != 1 or projected_features.shape[-1] != word_embeddings.shape[-1]:
        raise ValueError("图像数量或 hidden size 不匹配。")
    vectors = projected_features.reshape(-1, projected_features.shape[-1])
    if image_mask.sum().item() != vectors.shape[0]:
        raise ValueError("图像占位符数量必须与视觉向量数量相同。")
    fused = word_embeddings.clone()
    fused[image_mask] = vectors.to(device=fused.device, dtype=fused.dtype)
    return fused


In [ ]:
with capture_fusion(model.model) as captured:
    with torch.inference_mode():
        probe = model(**inputs_gpu, use_cache=False, logits_to_keep=1, return_dict=True)
assert len(captured["llm_calls"]) == 1
actual_fused_cpu = captured["llm_calls"][0]
manual_fused_cpu = reconstruct_fusion(word_embeddings_cpu, token_mask_cpu, captured["projected"])

# 同一次 forward 的 projector 输出用于重建，预期逐元素完全一致。
torch.testing.assert_close(manual_fused_cpu, actual_fused_cpu, rtol=0, atol=0)
torch.testing.assert_close(actual_fused_cpu[~token_mask_cpu], word_embeddings_cpu[~token_mask_cpu], rtol=0, atol=0)
torch.testing.assert_close(actual_fused_cpu[token_mask_cpu], captured["projected"].reshape(-1, actual_fused_cpu.shape[-1]), rtol=0, atol=0)
torch.testing.assert_close(captured["projected"], projected_reference, rtol=1e-2, atol=1e-2)
fusion_trace = {
    "input_ids": list(inputs_cpu["input_ids"].shape), "image_tokens": int(token_mask_cpu.sum()),
    "projector_input": captured["projector_input_shape"],
    "projector_output": list(captured["projected"].shape),
    "actual_llm_inputs_embeds": list(actual_fused_cpu.shape),
    "logits": list(probe.logits.shape),
    "manual_vs_actual_max_abs_diff": (manual_fused_cpu.float() - actual_fused_cpu.float()).abs().max().item(),
    "image_positions_equal_projector": True, "text_positions_unchanged": True,
}
save_json(RUN_DIR / "fusion-trace.json", fusion_trace)
print(json.dumps(fusion_trace, ensure_ascii=False, indent=2))
del probe, captured, actual_fused_cpu, manual_fused_cpu
print("现在用自己的话解释：为什么图像特征要映射到 text hidden size？")


## 7. 故意少放一个图像 token，观察一致性检查

这不是一次生成实验。只在局部构造错误的 ID 序列，并调用模型的占位符检查函数，预期得到数量不匹配的 `ValueError`。不会修改 processor、模型参数或正常输入。

思考：随意用 `max_length` 截断输入，为什么可能破坏图像 token 与视觉向量的对应关系？


In [ ]:
keep = torch.ones(inputs_cpu["input_ids"].shape[1], dtype=torch.bool)
keep[positions[0]] = False
bad_ids = inputs_cpu["input_ids"][:, keep].to(DEVICE)
with torch.inference_mode():
    bad_embeddings = model.get_input_embeddings()(bad_ids)
    try:
        model.model.get_placeholder_mask(
            bad_ids, inputs_embeds=bad_embeddings,
            image_features=projected_reference.to(DEVICE).reshape(-1, projected_reference.shape[-1]),
        )
    except ValueError as error:
        mismatch_result = {"expected_error": type(error).__name__, "message": str(error),
                           "image_tokens_after_removal": int(bad_ids.eq(image_token_id).sum()),
                           "visual_vectors": projected_reference.shape[1]}
    else:
        raise AssertionError("本次没有检测到预期错误，请检查版本与实验输入。")
del bad_ids, bad_embeddings
save_json(RUN_DIR / "placeholder-mismatch.json", mismatch_result)
print(json.dumps(mismatch_result, ensure_ascii=False, indent=2))


## 8. 跟踪 generate：首步 prefill 与后续解码

`forward` 一次得到下一 token 的分数；`generate` 管理多步生成、停止条件及 cache。下面只生成最多 8 个新 token，用 hook 记录视觉编码器调用次数与每次 LLM 输入形状，不复制整段隐藏状态。

通常首步处理完整图文序列，后续借助 KV cache 每步输入一个新 token。实际次数由模型停止位置决定；若提前 EOS，不应强行期待 8 步。这些形状只能说明调用行为，不能直接说明模型关注了哪里。


In [ ]:
def generate_answer(pil_image, question, generation=None):
    current_prompt, batch_cpu = prepare_inputs(pil_image, question)
    options = dict(GENERATION if generation is None else generation)
    torch.cuda.synchronize()
    torch.cuda.reset_peak_memory_stats()
    started = time.perf_counter()
    batch_gpu = to_device(batch_cpu)
    with torch.inference_mode():
        output_ids = model.generate(**batch_gpu, **options)
    torch.cuda.synchronize()
    elapsed = time.perf_counter() - started
    new_ids = output_ids[:, batch_cpu["input_ids"].shape[1]:].detach().cpu()
    result = {
        "raw_answer": processor.batch_decode(new_ids, skip_special_tokens=True)[0].strip(),
        "prompt": current_prompt, "new_token_ids": new_ids[0].tolist(),
        "new_tokens": new_ids.shape[1], "generation": options,
        "transfer_and_generate_seconds": elapsed,
        "peak_allocated_gib": torch.cuda.max_memory_allocated() / 1024**3,
        "peak_reserved_gib": torch.cuda.max_memory_reserved() / 1024**3,
    }
    # 这里的计时不含 prepare_inputs、图片打开和回答解码；不与第1章计时直接等同。
    del batch_gpu, output_ids
    return result

@contextmanager
def capture_generation(core):
    trace = {"vision_calls": 0, "projector_calls": 0, "llm_steps": []}
    handles = []

    def vision_hook(module, args, output):
        trace["vision_calls"] += 1

    def projector_hook(module, args, output):
        trace["projector_calls"] += 1

    def llm_pre_hook(module, args, kwargs):
        tensor = kwargs.get("inputs_embeds")
        if tensor is None:
            tensor = kwargs.get("input_ids")
        cache = kwargs.get("past_key_values")
        cache_length = int(cache.get_seq_length()) if hasattr(cache, "get_seq_length") else None
        trace["llm_steps"].append({"step": len(trace["llm_steps"]),
                                   "input_shape": list(tensor.shape) if tensor is not None else None,
                                   "past_sequence_length_before_forward": cache_length})
    try:
        handles.append(core.vision_tower.register_forward_hook(vision_hook))
        handles.append(core.multi_modal_projector.register_forward_hook(projector_hook))
        handles.append(core.language_model.register_forward_pre_hook(llm_pre_hook, with_kwargs=True))
        yield trace
    finally:
        for handle in handles:
            handle.remove()


In [ ]:
short_generation = {**GENERATION, "max_new_tokens": 8}
with capture_generation(model.model) as generation_trace:
    short_answer = generate_answer(image, QUESTION, generation=short_generation)
assert generation_trace["llm_steps"], "没有捕获到语言模型调用。"
assert generation_trace["llm_steps"][0]["input_shape"][1] == inputs_cpu["input_ids"].shape[1]
save_json(RUN_DIR / "generation-trace.json", {"trace": generation_trace, "result": short_answer})
print(json.dumps(generation_trace, ensure_ascii=False, indent=2))
print("短输出（可能因 8-token 上限不完整）:", short_answer["raw_answer"])
print("思考：视觉编码器调用了几次？后续步的输入长度为何与首步不同？")


## 9. 做有背景对照的遮挡实验

固定图片、问题、统一指令及生成配置，比较：原图、工作人员区域遮挡、等面积背景区域遮挡。下面默认坐标**仅针对 640×459 的样例图片**。换图时必须重新设置；先看框的位置，再运行推理格。

遮挡框采用半开区间 `(左,上,右,下)`，填充为同样的灰色；两框面积相同，但画面内容不同，不能消除所有混杂。框内外可能仍有相关颜色信息，请自行说明哪些证据还可见。每次执行预览格建立独立子目录；推理格重跑会创建新的 JSONL，不覆盖已有记录。

本章使用中性题干且统一追加指令，与第1章的单样本遮挡设置不同，因此本节是新的对照，不能直接把两章实验的回答差异归因于模型变化。


In [ ]:
def validate_equal_boxes(boxes, size):
    width, height = size
    areas = []
    for name, box in boxes.items():
        if len(box) != 4 or any(not isinstance(v, int) for v in box):
            raise ValueError(f"{name} 必须是 4 个整数坐标。")
        left, top, right, bottom = box
        if not (0 <= left < right <= width and 0 <= top < bottom <= height):
            raise ValueError(f"{name} 越界或面积为零：{box}, image={size}")
        areas.append((right - left) * (bottom - top))
    if len(set(areas)) != 1:
        raise ValueError("关键区域与背景区域必须使用相同面积。")
    return areas[0]

BOXES = {"target_occluded": (320, 210, 490, 355), "background_occluded": (10, 10, 180, 155)}
mask_area = validate_equal_boxes(BOXES, image.size)
EXPERIMENT_DIR = RUN_DIR / ("occlusion-" + stamp())
EXPERIMENT_DIR.mkdir()
variants = {"original": image.copy()}
preview = image.copy()
draw = ImageDraw.Draw(preview)
for name, box in BOXES.items():
    variant = image.copy()
    variant.paste((128, 128, 128), box)
    variants[name] = variant
    draw.rectangle(box, outline="red" if name == "target_occluded" else "yellow", width=3)
display(preview)
for name, variant in variants.items():
    print(name)
    display(variant)
    variant.save(EXPERIMENT_DIR / f"{name}.png")
save_json(EXPERIMENT_DIR / "design.json", {
    "source_image": str(IMAGE_PATH), "source_sha256": sha256_file(IMAGE_PATH),
    "boxes": BOXES, "mask_area_pixels": mask_area, "fill_rgb": [128, 128, 128],
    "question": QUESTION, "instruction": QA_INSTRUCTION, "generation": GENERATION,
})
print("确认红框覆盖目标、黄框位于背景后，再执行下一格。")


In [ ]:
REPEATS = 1  # 可改为 3 观察重复运行；贪心生成的重复不构成独立样本统计。
assert isinstance(REPEATS, int) and REPEATS >= 1
occlusion_path = EXPERIMENT_DIR / ("predictions-" + stamp() + ".jsonl")
occlusion_results = []
with occlusion_path.open("x", encoding="utf-8") as stream:
    for repeat in range(REPEATS):
        for name, variant in variants.items():
            row = {"variant": name, "repeat": repeat, "score": None,
                   "human_notes": "", "error": None,
                   "saved_image": f"{name}.png",
                   "saved_image_sha256": sha256_file(EXPERIMENT_DIR / f"{name}.png")}
            try:
                row.update(generate_answer(variant, QUESTION))
            except Exception as error:
                row["error"] = f"{type(error).__name__}: {error}"
                stream.write(json.dumps(row, ensure_ascii=False) + "\n")
                stream.flush()
                raise  # 停止本轮，不把运行失败作为模型回答进行比较。
            occlusion_results.append(row)
            stream.write(json.dumps(row, ensure_ascii=False) + "\n")
            stream.flush()
            print(name, "→", row["raw_answer"])
print("结果:", occlusion_path)
print("请记录：是否答蓝色 / 表示无法确定？背景对照是否变化？遮挡后还剩哪些可见证据？")


## 10. 用小张量理解两类连接方式（CPU 教学示意）

LLaVA 本次路径对所选 patch 向量逐个做投影，数量不变。MiniGPT-4 的经典 Q-Former 路径先用固定数量的 query 提取视觉信息，再映射到语言模型维度；具体走哪条分支取决于所选源码与配置，不能把所有 MiniGPT 变体当成同一个结构。

下面使用随机权重和缩小的隐藏维度演示“保留 patch 数量”与“通过 query 汇聚”的区别。**它不是 Q-Former 的复现，也不是训练或性能比较**；原模型的层数、预训练权重及完整处理过程不在这个玩具模块中。

参考：[MiniGPT4.encode_img](https://github.com/Vision-CAIR/MiniGPT-4/blob/main/minigpt4/models/minigpt4.py)。源码含有 `has_qformer` 分支，下一节记录实际 commit 再阅读。


In [ ]:
def connector_shape_demo():
    # fork_rng 使这个 CPU 小实验不改变后续实验的 CPU 随机状态。
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(7)
        patches = torch.randn(1, 576, 64)
        per_patch_projector = torch.nn.Sequential(torch.nn.Linear(64, 128), torch.nn.GELU(), torch.nn.Linear(128, 128))
        queries = torch.randn(1, 32, 64)
        attention = torch.nn.MultiheadAttention(64, num_heads=4, batch_first=True)
        query_projector = torch.nn.Linear(64, 128)
        with torch.inference_mode():
            patch_output = per_patch_projector(patches)
            query_features, _ = attention(queries, patches, patches, need_weights=False)
            query_output = query_projector(query_features)
    assert patch_output.shape == (1, 576, 128)
    assert query_output.shape == (1, 32, 128)
    return {"demo_only": True, "input": list(patches.shape),
            "per_patch_projection": list(patch_output.shape),
            "query_aggregation_then_projection": list(query_output.shape)}

connector_demo = connector_shape_demo()
save_json(RUN_DIR / "connector-shape-demo.json", connector_demo)
print(json.dumps(connector_demo, ensure_ascii=False, indent=2))
print("思考：向量数量和每个向量的维度，分别由哪个模块决定？数量更少能直接推出精度更好吗？")


## 11. 阅读原生 LLaVA 与 MiniGPT-4 源码，记录版本（可选联网步骤）

核心张量实验只依赖当前已安装的 HF 模型。完成架构对照时，将 `FETCH_PUBLIC_SOURCES` 改为 True：只下载几个公开源码 / 配置文本文件，不安装依赖、不导入外部代码、不下载 MiniGPT-4 权重。先用 GitHub API 将分支解析成 commit，再按该 commit 取文件；把该次解析得到的 commit 填回 `SOURCE_REVISIONS`，下次即可固定版本。

文件缓存到忽略提交的 `external/source-readings/`，索引保存到本次 `outputs/`。如果 GitHub 不通或限流，此节会记录失败，核心实验仍可保留；不要把下载失败当成对照已完成。也可以自行阅读对应官方仓库，在学习记录中填写实际 commit 与证据。

阅读时回答：

1. 原生 LLaVA 的 `encode_images`、`prepare_inputs_labels_for_multimodal` 与 HF 本次调用链分别在哪里完成投影和插入？
2. MiniGPT-4 的所选配置是否使用 Q-Former？query 数来自哪里？`llama_proj` 的输入是什么？
3. `minigpt_base.py` 的 `get_context_emb` 如何把图像 embedding 放到文本之间？
4. 配置和初始化源码如何设置冻结？不要把推理时 `eval()` 或 `inference_mode()` 等同于训练时冻结参数。


In [ ]:
import re
import urllib.request

FETCH_PUBLIC_SOURCES = False
SOURCE_REVISIONS = {"llava": None, "minigpt4": None}  # 首次联网解析；之后填实际 40 位 commit
SOURCE_SPECS = {
    "llava": {"repo": "haotian-liu/LLaVA", "files": [
        "llava/model/llava_arch.py", "llava/model/multimodal_projector/builder.py"]},
    "minigpt4": {"repo": "Vision-CAIR/MiniGPT-4", "files": [
        "minigpt4/models/minigpt4.py", "minigpt4/models/minigpt_base.py",
        "minigpt4/configs/models/minigpt4_vicuna0.yaml"]},
}

def fetch_text(url):
    request = urllib.request.Request(url, headers={"User-Agent": "MLLM-learning-chapter2"})
    with urllib.request.urlopen(request, timeout=30) as response:
        return response.read().decode("utf-8")

comparison_sources = {}
if FETCH_PUBLIC_SOURCES:
    for name, spec in SOURCE_SPECS.items():
        record = {"repository": spec["repo"], "files": [], "status": "pending"}
        comparison_sources[name] = record
        try:
            revision = SOURCE_REVISIONS[name]
            if revision is None:
                revision = json.loads(fetch_text(f"https://api.github.com/repos/{spec['repo']}/commits/main"))["sha"]
            if re.fullmatch(r"[0-9a-f]{40}", revision) is None:
                raise ValueError("源码 revision 必须是完整的 40 位 commit。")
            record["commit"] = revision
            for relative_path in spec["files"]:
                local_path = PROJECT_ROOT / "external/source-readings" / name / revision / relative_path
                url = f"https://raw.githubusercontent.com/{spec['repo']}/{revision}/{relative_path}"
                if not local_path.exists():
                    text = fetch_text(url)
                    local_path.parent.mkdir(parents=True, exist_ok=True)
                    with local_path.open("x", encoding="utf-8", newline="\n") as output:
                        output.write(text)
                record["files"].append({"path": str(local_path.relative_to(PROJECT_ROOT)),
                                         "url": url, "sha256": sha256_file(local_path)})
            record["status"] = "downloaded_for_reading"
        except Exception as error:
            record["status"] = "failed"
            record["error"] = f"{type(error).__name__}: {error}"
        save_json(RUN_DIR / "comparison-source-index.json", comparison_sources)
    print(json.dumps(comparison_sources, ensure_ascii=False, indent=2))
else:
    print("尚未下载源码。需要做版本化对照时，将 FETCH_PUBLIC_SOURCES 设为 True 再运行此格。")


In [ ]:
def show_source_windows(path, needles, radius=3, max_lines=100):
    lines = Path(path).read_text(encoding="utf-8").splitlines()
    selected = set()
    for number, line in enumerate(lines):
        if any(needle in line for needle in needles):
            selected.update(range(max(0, number - radius), min(len(lines), number + radius + 1)))
    for number in sorted(selected)[:max_lines]:
        print(f"{number + 1:4}: {lines[number]}")
    print("完整文件:", path)

for name, record in comparison_sources.items():
    if record["status"] != "downloaded_for_reading":
        continue
    print("\n项目:", name, "commit:", record["commit"])
    for entry in record["files"]:
        print("\n", entry["url"])
        show_source_windows(PROJECT_ROOT / entry["path"], [
            "def encode_images", "def prepare_inputs_labels_for_multimodal", "def encode_img",
            "has_qformer", "num_query_token", "llama_proj", "query_output",
            "def get_context_emb", "torch.cat", "freeze_", "requires_grad", "projector_type",
        ])


## 12. 生成证据索引与待填写的学习记录

本格只汇总已经存在的文件与断言结果，不自动声称本章完成。遮挡结论、源码比较和未解决问题仍需你填写。重复执行会生成新文件，保留之前的记录。

完成标准：能结合 `token-trace.json`、`vision-trace.json`、`fusion-trace.json`、`generation-trace.json` 解释图像从哪里进入 LLM；说明文本位置为什么保持不变；完成遮挡对照及选定源码版本的连接模块比较。


In [ ]:
record_files = ["config.json", "environment.json", "requirements-freeze.txt", "model-config.json",
                "generation-config.json", "processor-config.json", "token-trace.json",
                "installed-source-index.json", "vision-trace.json", "fusion-trace.json",
                "placeholder-mismatch.json", "generation-trace.json", "connector-shape-demo.json",
                "comparison-source-index.json"]
evidence = []
for name in record_files:
    path = RUN_DIR / name
    if path.is_file():
        evidence.append({"path": name, "sha256": sha256_file(path)})
occlusion_outputs = sorted(RUN_DIR.glob("occlusion-*/predictions-*.jsonl"))
for path in occlusion_outputs:
    evidence.append({"path": str(path.relative_to(RUN_DIR)), "sha256": sha256_file(path)})
for path in sorted(RUN_DIR.glob("occlusion-*/*")):
    if path.is_file() and path.suffix in {".png", ".json"}:
        evidence.append({"path": str(path.relative_to(RUN_DIR)), "sha256": sha256_file(path)})

source_downloads_complete = (bool(comparison_sources) and all(
    record["status"] == "downloaded_for_reading" for record in comparison_sources.values()))
summary = {"model_id": MODEL_ID, "model_revision": MODEL_REVISION,
           "image_tokens": expected_image_tokens, "vision": vision_trace, "fusion": fusion_trace,
           "generation": generation_trace, "occlusion_prediction_files": [str(p.relative_to(RUN_DIR)) for p in occlusion_outputs],
           "comparison_sources_downloaded": source_downloads_complete,
           "comparison_reading_completed": None, "chapter2_completed": None,
           "note": "下载成功不等于阅读完成；人工结论待填写。", "evidence": evidence}
summary_path = RUN_DIR / ("learning-summary-" + stamp() + ".json")
save_json(summary_path, summary)

note_lines = [
    "# 第2章学习记录（待填写）", "",
    f"- 模型：{MODEL_ID} @ {MODEL_REVISION}", f"- 运行目录：{RUN_DIR}",
    f"- 图像 token：{expected_image_tokens}",
    f"- 真实 LLM 输入：{fusion_trace['actual_llm_inputs_embeds']}",
    f"- 手工融合与真实输入最大绝对差：{fusion_trace['manual_vs_actual_max_abs_diff']}", "",
    "## 我的解释", "",
    "1. 模板的一个 <image> 为什么变成多个 token？待填写。",
    "2. 视觉特征经过哪些函数进入 LLM？待填写。",
    "3. 图像位置与文本位置分别如何核对？待填写。",
    "4. prefill 与 decode 的输入长度、视觉调用次数如何变化？待填写。", "",
    "## 遮挡对照", "",
    "原图 / 关键区域 / 等面积背景的回答、仍可见的信息、局限：待填写。", "",
    "## 源码连接模块对比", "",
    "HF LLaVA / 原生 LLaVA / 所选 MiniGPT-4 配置、commit、函数、向量数量与维度、冻结策略：待填写。", "",
    "## 尚未验证的问题与下一步", "", "待填写。", "",
]
note_path = RUN_DIR / ("learning-notes-" + stamp() + ".md")
note_path.write_text("\n".join(note_lines), encoding="utf-8")
print("证据摘要:", summary_path)
print("学习记录:", note_path)
print("原始输出保留在 outputs；填写完成后，把精简结论整理到 notes/ 或 experiments/，再更新 index.md。")


## 保存与常见问题

- **先保存证据，再清空 notebook 输出。** 云端实例释放前备份 `outputs/ch2-.../`；Git 不会上传图片、原始预测或外部源码缓存。正式 notebook 提交前清空输出。
- **HF 缓存未命中 / 网络失败**：检查第 0 节 endpoint、实际缓存路径及 `LOCAL_FILES_ONLY`，不要改模型 commit 来回避问题。
- **GPU OOM**：确认第1章内核已关闭；重启后从头运行。本 notebook batch=1、仅保存必要 CPU 副本，不请求 LLM 的全部层隐藏状态或 attention。
- **shape / token 数不匹配**：先核对模型 revision、Transformers 4.57.1、processor patch size、CLS 与 feature strategy；不要直接删掉断言。
- **hook 报错**：hook 会由 finally 清理；修复后重跑该实验格，不必重新加载模型。
- **重复执行第 0 格报错**：这是导入顺序保护；要改缓存或 endpoint 时重启内核。
- **MiniGPT-4 下载失败**：核心 HF 数据流实验不依赖这一步；记录未完成的源码对照，联网恢复后继续。不要安装完整 MiniGPT-4 来解决纯源码阅读问题。

完成后应能不看代码回答：**图像 token 的 ID 是图像内容吗？projector 改变的是数量还是维度？LLM 实际收到什么？为什么不在每个生成步重新编码图片？**
